PLEASE REFER TO APPENDIX ON HOW TO RUN THE CODE, CHANGE THE FILE NAMES ACCORDINGLY IN ORDER FOR THE DATASETS TO BE IMPORTED ON YOUR SYSTEM. YOU CAN LOCATE THE WHEREABOUTS OF THE CSV'S BY CHECKING THE PATHWAY WHERE THE CODE IS TRYING TO IMPORT


USE localized_url function if there's an error with the imports

In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [71]:
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf
import xgboost as xgb
import numpy as np
import pandas as pd
import random
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')  # Suppress Python warnings


random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
import os
def localized_url(url):

    current_dir = os.getcwd().split("statistical-analysis-and-modeling-of-football-data")[0] + "statistical-analysis-and-modeling-of-football-data"
    branch = url.split("statistical-analysis-and-modeling-of-football-data")[1]
    return current_dir + branch

In [ ]:
team_df= pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Aggregation/Data/under_team_pivoted.csv'))
fifa_df= pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Fifa/Data/fifa_stats_fully_processed.csv'))
elo_df = pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/ClubELO/Data/understat_team_elo_with_seasonal_elo.csv'))


roster_aggregated_df = pd.read_csv(localized_url('/Users/yarkin/Desktop/Dissertation/statistical-analysis-and-modeling-of-football-data/Aggregation/Data/understat_roster_aggregated.csv'))

In [73]:
roster_aggregated_df.columns

Index(['goals', 'own_goals', 'shots', 'xG', 'time', 'player_id', 'team_id',
       'position', 'player', 'h_a', 'yellow_card', 'red_card', 'key_passes',
       'assists', 'xA', 'xGChain', 'xGBuildup', 'positionOrder', 'match_link',
       'id', 'date', 'shots_median', 'shots_median_error', 'xG_median',
       'xG_median_error', 'key_passes_median', 'key_passes_median_error',
       'xA_median', 'xA_median_error', 'xGChain_median',
       'xGChain_median_error', 'xGBuildup_median', 'xGBuildup_median_error',
       'yellow_card_median', 'yellow_card_median_error', 'red_card_median',
       'red_card_median_error'],
      dtype='object')

In [74]:
roster_aggregated_df = roster_aggregated_df[['shots', 'xG', 'team_id',
       'h_a', 'yellow_card', 'red_card', 'key_passes', 'xA', 'xGChain', 'xGBuildup',
       'id', 'date', 'shots_median', 'xG_median', 'key_passes_median',
       'xA_median', 'xGChain_median','xGBuildup_median','yellow_card_median', 'red_card_median']]

columns_to_sum = [ 'shots', 'xG', 'yellow_card', 'red_card', 'key_passes',
       'xA', 'xGChain', 'xGBuildup', 'shots_median', 'xG_median', 'key_passes_median',
       'xA_median', 'xGChain_median', 'xGBuildup_median', 'yellow_card_median',
       'red_card_median']

In [75]:
aggregated_df = roster_aggregated_df.groupby(['id', 'team_id', 'h_a'])[columns_to_sum].sum().reset_index()


In [76]:
aggregated_df

,id,team_id,h_a,shots,xG,yellow_card,red_card,key_passes,xA,xGChain,xGBuildup,shots_median,xG_median,key_passes_median,xA_median,xGChain_median,xGBuildup_median,yellow_card_median,red_card_median
0,81,82,a,9,0.675723,3,0,7,0.586365,1.745371,0.811549,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0
1,81,89,h,9,0.627539,2,0,5,0.284979,1.396328,0.994160,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0
2,82,71,a,7,0.786230,4,0,4,0.560695,1.238205,0.736815,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0
3,82,73,h,11,0.876106,3,0,9,0.419975,2.159510,1.170895,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0
4,83,72,h,10,0.604226,1,0,8,0.549139,1.025550,0.493522,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6835,22303,238,h,6,0.762232,4,0,3,0.674501,1.963836,1.189961,7.0,0.460030,4.0,0.255956,1.031553,0.448818,1.0,0.0
6836,22304,86,h,23,1.622070,1,0,15,0.890880,5.184757,3.304182,10.0,1.390762,5.0,0.573263,5.429534,3.474101,1.0,0.0
6837,22304,87,a,9,0.856121,2,1,7,0.783602,2.174675,0.917865,12.0,1.276066,8.0,1.113011,4.095123,2.293432,0.0,0.0
6838,25861,73,h,24,2.874502,1,0,22,2.191435,7.485275,4.355573,17.0,2.199968,6.0,0.461495,3.412833,1.541264,0.0,0.0


In [77]:
elo_df = elo_df[['id', 'Elo_team_h',  'Elo_team_a', 'Seasonal_Elo_Home',
       'Seasonal_Elo_Away']].copy()
elo_df["ClubELO"] = elo_df["Elo_team_h"] - elo_df["Elo_team_a"]
elo_df["Seasonal_Elo"] = elo_df["Seasonal_Elo_Home"] - elo_df["Seasonal_Elo_Away"]
elo_df = elo_df[['id', 'ClubELO', 'Seasonal_Elo']].copy()

In [78]:
# Merge aggregated_df with elo_df, elo_df will be + if h_a == 'h' and - if h_a == 'a'
merged_df = pd.merge(aggregated_df, elo_df, on='id', how='left')
merged_df['ClubELO'] = np.where(merged_df['h_a'] == 'h', merged_df['ClubELO'], -merged_df['ClubELO'])
merged_df['Seasonal_Elo'] = np.where(merged_df['h_a'] == 'h', merged_df['Seasonal_Elo'], -merged_df['Seasonal_Elo'])

In [79]:
fifa_df.drop(columns=['yellow_card', 'red_card', 'key_passes', 'assists', 'xG',
       'xGBuildup'], inplace=True)

In [80]:
fifa_df.columns

Index(['id', 'Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj'],
      dtype='object')

In [81]:
fifa_stats = ['Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj']

In [82]:
merged_df

,id,team_id,h_a,shots,xG,yellow_card,red_card,key_passes,xA,xGChain,...,shots_median,xG_median,key_passes_median,xA_median,xGChain_median,xGBuildup_median,yellow_card_median,red_card_median,ClubELO,Seasonal_Elo
0,81,82,a,9,0.675723,3,0,7,0.586365,1.745371,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,-79.658081,-57.467305
1,81,89,h,9,0.627539,2,0,5,0.284979,1.396328,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,79.658081,57.467305
2,82,71,a,7,0.786230,4,0,4,0.560695,1.238205,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,-16.700806,-21.339912
3,82,73,h,11,0.876106,3,0,9,0.419975,2.159510,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,16.700806,21.339912
4,83,72,h,10,0.604226,1,0,8,0.549139,1.025550,...,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.0,132.473877,39.314839
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6835,22303,238,h,6,0.762232,4,0,3,0.674501,1.963836,...,7.0,0.460030,4.0,0.255956,1.031553,0.448818,1.0,0.0,-451.948853,-162.614728
6836,22304,86,h,23,1.622070,1,0,15,0.890880,5.184757,...,10.0,1.390762,5.0,0.573263,5.429534,3.474101,1.0,0.0,-62.010986,-10.819436
6837,22304,87,a,9,0.856121,2,1,7,0.783602,2.174675,...,12.0,1.276066,8.0,1.113011,4.095123,2.293432,0.0,0.0,62.010986,10.819436
6838,25861,73,h,24,2.874502,1,0,22,2.191435,7.485275,...,17.0,2.199968,6.0,0.461495,3.412833,1.541264,0.0,0.0,89.269897,116.813483


In [83]:
# Merge merged DataFrame with fifa_df, fifa_df will be + if h_a == 'h' and - if h_a == 'a'
merged_df = pd.merge(merged_df, fifa_df, on='id', how='left')



In [84]:
merged_df[fifa_stats] = merged_df[fifa_stats].where(
    merged_df['h_a'] == 'h',
    -merged_df[fifa_stats]
)

In [85]:
# Target can be any of these
target = "xG"

# All inputs (choose either overalls or full FIFA features)
feature_cols = [
    # Aggregated values (example set, expand if needed)
    'shots_median', 'xG_median', 'key_passes_median',
    'xA_median', 'xGChain_median', 'xGBuildup_median', 'yellow_card_median',
    'red_card_median',

    # Elo
    "ClubELO", "Seasonal_Elo",

    #  full FIFA features
    'Overall', 'Finishing_adj', 'Shot Power_adj', 'Long Shots_adj',
       'Heading_adj', 'Curve_adj', 'FK Acc._adj', 'Penalties_adj',
       'Short Pass_adj', 'Long Pass_adj', 'Stand Tackle_adj',
       'Interceptions_adj', 'Jumping_adj', 'Stamina_adj', 'Strength_adj',
       'Acceleration_adj', 'Agility_adj', 'Balance_adj', 'GK Handling_adj',
       'GK Kicking_adj', 'GK Positioning_adj', 'Reactions_adj',
       'Att. Position_adj', 'Vision_adj', 'Dribbling_adj',
       'Overall Passing_adj', 'Overall Shooting_adj', 'Overall Defence_adj',
       'Overall Physical_adj', 'Overall Goalkeeper_adj']


In [86]:
def eval_random_forest(X_train, y_train, X_test, y_test, feature_cols):
    model = RandomForestRegressor(n_estimators=300, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    importances = model.feature_importances_
    feature_importance_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
    most_important_5_list = feature_importance_df.nlargest(5, 'importance').to_dict('records')
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), most_important_5_list

def eval_neural_net(X_train, y_train, X_test, y_test):
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_tr_scaled.shape[1],)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(X_tr_scaled, y_tr, validation_data=(X_val_scaled, y_val),
              epochs=500, batch_size=32, callbacks=[es], verbose=0)
    model.scaler = scaler  # Store the scaler in the model for future use
    preds = model.predict(X_test_scaled, verbose=0).ravel()
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), None

def eval_xgboost(X_train, y_train, X_test, y_test):
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    dte  = xgb.DMatrix(X_test, label=y_test)

    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42
    }
    model = xgb.train(params, dtr, num_boost_round=5000,
                      evals=[(dtr, 'train'), (dval, 'val')],
                      early_stopping_rounds=200, verbose_eval=False)
    preds = model.predict(dte, iteration_range=(0, model.best_iteration+1))
    return mean_absolute_error(y_test, preds), r2_score(y_test, preds), None

def try_models(df, feature_cols, target, full=True, n_splits=5):
    X = df[feature_cols].values
    y = df[target].values
    tscv = TimeSeriesSplit(n_splits=n_splits)

    all_results = []
    splits = list(tscv.split(X))  # cache splits

    for fold, (train_idx, test_idx) in enumerate(splits, 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        for name, func in [
            ("Random Forest", lambda X_tr, y_tr, X_te, y_te: eval_random_forest(X_tr, y_tr, X_te, y_te, feature_cols)),
            ("Neural Network", eval_neural_net),
            ("XGBoost", eval_xgboost)
        ]:
            mae, r2, top_features = func(X_train, y_train, X_test, y_test)
            all_results.append({
                'Fold': fold,
                'Target': target,
                'Model': name,
                'MAE': mae,
                'R2': r2,
                'Feature Types': 'Full' if full else 'Overall',
                'Important Features Dictionary': top_features
            })

    results_df = pd.DataFrame(all_results)

    perf_df = results_df.groupby(
        ["Target", "Model", "Feature Types"], as_index=False
    ).agg({
        "MAE": "mean",
        "R2": "mean",
        "Important Features Dictionary": lambda x: [d for d in x if d is not None]
    })

    # Train each model on the last fold for return values
    train_idx, test_idx = splits[-1]
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Random Forest
    rf_model = RandomForestRegressor(n_estimators=300, random_state=42).fit(X_train, y_train)

    # Neural Net
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    nn_model = Sequential([
        Dense(128, activation='relu', input_shape=(X_tr_scaled.shape[1],)),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    nn_model.compile(optimizer='adam', loss='mean_squared_error')
    nn_es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    nn_model.fit(X_tr_scaled, y_tr, validation_data=(X_val_scaled, y_val),
                 epochs=500, batch_size=32, callbacks=[nn_es], verbose=0)
    nn_model.scaler = scaler
    # XGBoost
    X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, shuffle=False)
    dtr = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    xgb_params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'mae',
        'max_depth': 6,
        'eta': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'seed': 42
    }
    xgb_model = xgb.train(
        xgb_params, dtr, num_boost_round=5000,
        evals=[(dtr, 'train'), (dval, 'val')],
        early_stopping_rounds=200, verbose_eval=False
    )

    return rf_model, nn_model, xgb_model, perf_df


In [87]:
performance_df = pd.DataFrame()
full_models_dict = {}

In [88]:
target_list= [
    'shots', 'xG', 'yellow_card', 'red_card', 'key_passes',
       'xA', 'xGChain', 'xGBuildup'
]

In [89]:
for target in tqdm(target_list, desc="Processing targets"):
    rf_model, nn_model, xgb_model, perf_df = try_models(
        merged_df, feature_cols, target, full=True, n_splits=5
    )
    full_models_dict[target] = {
        'Random Forest': rf_model,
        'Neural Network': nn_model,
        'XGBoost': xgb_model
    }

    performance_df = pd.concat([performance_df, perf_df], ignore_index=True)

Processing targets: 100%|██████████| 8/8 [27:38<00:00, 207.31s/it]


In [90]:
performance_df["Important Features"]= "All"

In [91]:
performance_df

,Target,Model,Feature Types,MAE,R2,Important Features Dictionary,Important Features
0,shots,Neural Network,Full,3.886691,0.184356,[],All
1,shots,Random Forest,Full,3.683740,0.284426,"[[{'feature': 'ClubELO', 'importance': 0.17704...",All
2,shots,XGBoost,Full,3.726462,0.265035,[],All
3,xG,Neural Network,Full,0.633178,0.166317,[],All
4,xG,Random Forest,Full,0.616462,0.239255,"[[{'feature': 'Seasonal_Elo', 'importance': 0....",All
5,xG,XGBoost,Full,0.619646,0.227173,[],All
6,yellow_card,Neural Network,Full,1.051264,-0.089622,[],All
7,yellow_card,Random Forest,Full,1.017395,-0.000385,"[[{'feature': 'Seasonal_Elo', 'importance': 0....",All
8,yellow_card,XGBoost,Full,1.021422,-0.004221,[],All
9,red_card,Neural Network,Full,0.110620,-0.036201,[],All


In [92]:

from collections import defaultdict

def aggregate_rf_importances(perf_df, target, feature_type, model="Random Forest",
                             top_k=10, agg="sum", zero_fill_mean=False):
    """
    Aggregate per-fold RF top-k importances into one ranked table.

    agg: "sum" -> sum across folds
         "mean" -> mean across occurrences
         "mean_zero" -> mean across all folds, treating missing folds as zero
    zero_fill_mean: deprecated; use agg="mean_zero" instead
    """
    # Select the cell
    mask = (
        (perf_df['Target'] == target) &
        (perf_df['Feature Types'] == feature_type) &
        (perf_df['Model'] == model)
    )
    if not mask.any():
        raise ValueError("No matching row in perf_df for the given filters.")
    cell = perf_df.loc[mask, 'Important Features Dictionary'].values[0]

    # Count folds represented in the cell
    if isinstance(cell, list) and cell and isinstance(cell[0], list):
        n_folds = len(cell)
        nested_lists = cell
    else:
        # Already flattened to a single list (or None)
        n_folds = 1 if isinstance(cell, list) else 0
        nested_lists = [cell] if isinstance(cell, list) else []

    # Flatten to records [{'feature':..., 'importance':...}, ...]
    records = []
    for fold_list in nested_lists:
        if not isinstance(fold_list, list):
            continue
        for rec in fold_list:
            if isinstance(rec, dict) and 'feature' in rec and 'importance' in rec:
                records.append(rec)

    if not records:
        return pd.DataFrame(columns=['feature', 'score', 'occurrence', 'rank'])

    # Aggregate
    totals = defaultdict(float)
    counts = defaultdict(int)
    for rec in records:
        f = rec['feature']
        imp = float(rec['importance'])
        totals[f] += imp
        counts[f] += 1

    rows = []
    for f, total in totals.items():
        if agg == "sum":
            score = total
        elif agg == "mean":
            score = total / counts[f]
        elif agg == "mean_zero":
            # treat missing folds as zero (uses n_folds)
            score = total / max(n_folds, 1)
        else:
            raise ValueError("agg must be one of {'sum','mean','mean_zero'}")
        rows.append({'feature': f, 'score': score, 'occurrence': counts[f]})

    out = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    out['rank'] = np.arange(1, len(out) + 1)
    return out.head(top_k)


In [93]:
# If your variable is named performances_df (note the plural), pass that:
for target in target_list:
    print("Target:", target, "Full Features")
    print(aggregate_rf_importances(performance_df, target=target, feature_type="Full",
                                        model="Random Forest", top_k=10, agg="sum")[["feature", "score"]])




Target: shots Full Features
            feature     score
0      Seasonal_Elo  0.921318
1           ClubELO  0.556821
2           Overall  0.177398
3      shots_median  0.145147
4  Stand Tackle_adj  0.029343
5       Heading_adj  0.025865
6      Strength_adj  0.025849
7    Long Shots_adj  0.025769
8  Acceleration_adj  0.022080
Target: xG Full Features
            feature     score
0      Seasonal_Elo  1.051728
1           ClubELO  0.433186
2           Overall  0.166805
3     Reactions_adj  0.113811
4    xGChain_median  0.090455
5         xA_median  0.029167
6      shots_median  0.028516
7  xGBuildup_median  0.027689
Target: yellow_card Full Features
                  feature     score
0            Seasonal_Elo  0.268085
1           Reactions_adj  0.172467
2           Finishing_adj  0.167596
3         GK Handling_adj  0.062169
4          Short Pass_adj  0.061806
5             Balance_adj  0.060610
6                 ClubELO  0.060358
7  Overall Goalkeeper_adj  0.031567
8                 O

In [94]:
top_features_dict ={"shots":[
      "Seasonal_Elo",  #0.921318
      "ClubELO",  #0.556821
      "Overall",  #0.177398
      "shots_median",  #0.145147
      "Stand Tackle_adj",  #0.029343
      "Heading_adj",  #0.025865
      "Strength_adj",  #0.025849
      "Long Shots_adj"  #0.025769
] }

In [95]:

top_features_dict["xG"] = [
      "Seasonal_Elo",  #1.051728
      "ClubELO",  #0.433186
      "Overall",  #0.166805
      "Reactions_adj",  #0.113811
      "xGChain_median"  #0.090455
]

top_features_dict["yellow_card"] = [
    "Seasonal_Elo",  #0.268085
    "Reactions_adj",  #0.172467
    "Finishing_adj",  #0.167596
    "GK Handling_adj",  #0.062169
    "Short Pass_adj",  #0.061806
    "Balance_adj",  #0.060610
    "ClubELO"  #0.060358
]



top_features_dict["red_card"] = [
    "Long Pass_adj",  #0.213845
    "Reactions_adj",  #0.209997
    "Overall Shooting_adj",  #0.149560
    "Overall Physical_adj",  #0.078524
    "Jumping_adj",  #0.077393
    "Dribbling_adj"  #0.065426
]

top_features_dict["key_passes"] = [
    "Seasonal_Elo",  #0.726174
    "ClubELO",  #0.673306
    "Overall",  #0.213835
    "shots_median",  #0.113972
    "xGBuildup_median"  #0.060543
]


top_features_dict["xA"] = [
    "Seasonal_Elo",  #0.904339
    "ClubELO",  #0.439170
    "Overall",  #0.191383
    "xGChain_median",  #0.092693
    "Reactions_adj",  #0.059202
    "Overall Passing_adj"  #0.028506
]
top_features_dict["xGChain"] = [
    "ClubELO",  #0.996639
    "Seasonal_Elo",  #0.405047
    "xGChain_median",  #0.151748
    "Overall",  #0.147143
    "Reactions_adj"  #0.038875
]
top_features_dict["xGBuildup"] = [
    "ClubELO",  #0.919699
    "Seasonal_Elo",  #0.305191
    "xGBuildup_median",  #0.191377
    "Overall",  #0.142557
    "xGChain_median",  #0.116723
    "Reactions_adj",  #0.085290
    "Overall Passing_adj"  #0.065707
]


In [96]:
# --- Random Forest ---
def build_RF_small(df, target, top_features, test_size=0.2):
    X = df[top_features].values
    y = df[target].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, shuffle=False
    )
    rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    return rf, mae, r2

# --- Neural Network (fixed: no leakage) ---
def build_NN_small(df, target, top_features, test_size=0.2, val_size=0.2):
    X = df[top_features].values
    y = df[target].values
    # Hold-out test block
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, shuffle=False
    )
    # Validation from train block (no shuffling)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train, test_size=val_size, shuffle=False
    )

    scaler = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)
    X_val_s  = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_tr_s.shape[1],)),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

    model.fit(
        X_tr_s, y_tr,
        validation_data=(X_val_s, y_val),
        epochs=300, batch_size=32,
        callbacks=[es], verbose=0
    )

    y_pred = model.predict(X_test_s, verbose=0).ravel()
    mae = mean_absolute_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)

    # Attach scaler for downstream inference
    model.scaler = scaler
    return model, mae, r2

# --- Orchestrator ---
def get_small_models(df, top_features, target, full=True, n_splits=5):
    X = df[top_features].to_numpy()
    y = df[target].to_numpy()
    tscv = TimeSeriesSplit(n_splits=n_splits)

    perf = []
    for fold, (tr, te) in enumerate(tscv.split(X), 1):
        rf, mae_rf, r2_rf = build_RF_small(df.iloc[np.r_[tr, te]], target, top_features,
                                        test_size=len(te)/len(np.r_[tr, te]))
        nn, mae_nn, r2_nn = build_NN_small(df.iloc[np.r_[tr, te]], target, top_features,
                                        test_size=len(te)/len(np.r_[tr, te]))
        perf += [
            dict(Target=target, Model='Random Forest', MAE=mae_rf, R2=r2_rf,
                Fold=fold, Feature_Types='Full' if full else 'Overall'),
            dict(Target=target, Model='Neural Network', MAE=mae_nn, R2=r2_nn,
                Fold=fold, Feature_Types='Full' if full else 'Overall'),
        ]

    # --- average over folds & match old format ---
    perf_df = pd.DataFrame(perf)

    avg_df = (
        perf_df
        .groupby(['Target', 'Model', 'Feature_Types'], as_index=False)
        .agg(MAE=('MAE', 'mean'), R2=('R2', 'mean'))
        .rename(columns={'Feature_Types': 'Feature Types'})
    )

    # keep the same features column as before
    avg_df['Important Features'] = [top_features] * len(avg_df)

    # same columns/order as the old return
    avg_df = avg_df[['Target', 'Model', 'MAE', 'R2', 'Feature Types', 'Important Features']]

    return rf, nn, avg_df

In [97]:
full_models_dict_small = {}
for target in tqdm(target_list, desc="Processing small models"):
    top_features = top_features_dict[target]
    rf_model, nn_model, perf_df = get_small_models(
        merged_df, top_features, target, full=True
    )
    full_models_dict_small[target] = {
        'Random Forest': rf_model,
        'Neural Network': nn_model
    }
    performance_df = pd.concat([performance_df, perf_df], ignore_index=True)

Processing small models: 100%|██████████| 8/8 [04:14<00:00, 31.82s/it]


In [98]:
def return_best_row_for_all_targets(df, target_list):
    # Get the row with the maximum importance for the specified feature
    best_rows = pd.DataFrame()
    for target in target_list:
        best_row = df[df.Target == target].sort_values(by='MAE', ascending=True).head(1)
        best_rows = pd.concat([best_rows, best_row], ignore_index=True)
    best_rows = best_rows.reset_index(drop=True)
    return best_rows

In [99]:
best_rows_v3=return_best_row_for_all_targets(performance_df, target_list)[["Target", "Model", "MAE", "R2", "Feature Types", "Important Features"]]

In [100]:
best_rows_v3

,Target,Model,MAE,R2,Feature Types,Important Features
0,shots,Neural Network,3.672995,0.285918,Full,"[Seasonal_Elo, ClubELO, Overall, shots_median,..."
1,xG,Neural Network,0.612766,0.242615,Full,"[Seasonal_Elo, ClubELO, Overall, Reactions_adj..."
2,yellow_card,Random Forest,1.017395,-0.000385,Full,All
3,red_card,Neural Network,0.108731,-0.009085,Full,"[Long Pass_adj, Reactions_adj, Overall Shootin..."
4,key_passes,Random Forest,3.006762,0.274399,Full,All
5,xA,Neural Network,0.478713,0.207943,Full,"[Seasonal_Elo, ClubELO, Overall, xGChain_media..."
6,xGChain,Neural Network,2.060176,0.266330,Full,"[ClubELO, Seasonal_Elo, xGChain_median, Overal..."
7,xGBuildup,Neural Network,1.437128,0.244013,Full,"[ClubELO, Seasonal_Elo, xGBuildup_median, Over..."


In [102]:
# --- user-provided artifacts assumed to exist ---
# merged_df
# feature_cols
# top_features_dict
# full_models_dict, full_models_dict_small

# 1) Declare which model+feature scope to use for each target (from your message)
best_plan = {
    "shots":        ("Random Forest", "full"),
    "xG":           ("Neural Network", "small"),
    "yellow_card":  ("Neural Network", "small"),
    "red_card":     ("Neural Network", "full"),
    "key_passes":   ("Random Forest", "full"),
    "xA":           ("Neural Network", "small"),
    "xGChain":      ("Neural Network", "small"),
    "xGBuildup":    ("Neural Network", "small"),
}

# 2) Helper: choose model + features per target from the right dict
def _get_features_for(target, scope):
    if scope == "full":
        return feature_cols
    elif scope == "small":
        return top_features_dict[target]
    else:
        raise ValueError(f"Unknown scope {scope!r}")

def _get_model_for(target, model_name, scope):
    # "full" scope -> pick from full_models_dict
    # "small" scope -> pick from full_models_dict_small
    store = full_models_dict if scope == "full" else full_models_dict_small
    if target not in store or model_name not in store[target]:
        raise KeyError(f"Model for target={target}, model={model_name}, scope={scope} not found.")
    return store[target][model_name]

# 3) Unified prediction that handles NN scaler if present
def _predict_with_model(model, X):
    # Neural Net: use attached scaler if available
    if hasattr(model, "predict"):  # Keras or RF both have .predict
        # Keras model detection: presence of .scaler means small NN (or full NN after you attach it)
        scaler = getattr(model, "scaler", None)
        if scaler is not None:
            X = scaler.transform(X)
        else:
            # If it's a Keras NN without scaler (likely the 'full' NN from try_models),
            # we warn once and proceed unscaled (works, but not ideal).
            try:
                import tensorflow as tf  # cheap check that it's likely a Keras model
                if isinstance(model, tf.keras.Model):
                    print("[WARN] NN model has no attached scaler; using raw features for inference. "
                          "Attach scaler in try_models (nn_model.scaler = scaler) to match training.")
            except Exception:
                pass
        yhat = model.predict(X, verbose=0).ravel()
        return yhat
    else:
        # Shouldn't happen for sklearn/xgb/keras in your code paths
        raise TypeError("Unsupported model type; expected sklearn/xgb/keras with .predict")

# 4) Build all {target}_agg columns
for target, (model_name, scope) in best_plan.items():
    feats = _get_features_for(target, scope)
    mdl  = _get_model_for(target, model_name, scope)

    # Extract feature matrix in the same order
    X = merged_df[feats].to_numpy()

    # Predict
    if model_name == "Random Forest":
        preds = mdl.predict(X)
    else:  # Neural Network
        preds = _predict_with_model(mdl, X)

    # Save as new aggregated feature column
    merged_df[f"{target}_agg"] = preds

# 5) (Optional) quick sanity check printout
cols_preview = [f"{t}_agg" for t in best_plan.keys()]
print("Created columns:", cols_preview)
print(merged_df[cols_preview].head())


Created columns: ['shots_agg', 'xG_agg', 'yellow_card_agg', 'red_card_agg', 'key_passes_agg', 'xA_agg', 'xGChain_agg', 'xGBuildup_agg']
   shots_agg    xG_agg  yellow_card_agg  red_card_agg  key_passes_agg  \
0   8.973333  0.915552         2.144670      0.027815        7.110000   
1  10.466667  1.253127         2.006686      0.100558        6.383333   
2   8.170000  0.995381         1.647829      0.044423        5.500000   
3  11.520000  1.144858         1.591504      0.149994        8.660000   
4  11.793333  1.322937         1.360187      0.037222        8.933333   

     xA_agg  xGChain_agg  xGBuildup_agg  
0  0.673114     1.924609       1.126866  
1  0.896123     3.212300       1.795846  
2  0.766093     2.223085       1.251466  
3  0.802091     2.430154       1.320434  
4  0.955891     3.402409       1.953933  


In [103]:
# round shots, yellow_card, red_card, key_passes to close integers
merged_df['shots_agg'] = merged_df['shots_agg'].round().astype(int)
merged_df['yellow_card_agg'] = merged_df['yellow_card_agg'].round().astype(int)
merged_df['red_card_agg'] = merged_df['red_card_agg'].round().astype(int)
merged_df['key_passes_agg'] = merged_df['key_passes_agg'].round().astype(int)


In [104]:
# Save the final DataFrame with aggregated features
merged_df.to_csv("Data/understat_roster_model_aggregated.csv", index=False)